
# Anime Recommendation System

A content-based anime recommendation system built using Pandas, NumPy, and Scikit-learn.

The system combines genre similarity, synopsis similarity, and weighted ratings to recommend anime. It also supports franchise-aware recommendations to help users discover different series.



## 1. Imports and Dataset Loading


In [2]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
anime=pd.read_csv("anime.csv")
columns=anime.columns.to_list()


## 2. Initial Dataset Inspection


In [4]:
columns
anime.shape

(12294, 7)

In [5]:
print("Columns DType")
print(anime.dtypes)

Columns DType
anime_id      int64
name            str
genre           str
type            str
episodes        str
rating      float64
members       int64
dtype: object


In [6]:
print("null values")
print(anime.isnull().sum())
print(anime.shape)

null values
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64
(12294, 7)


In [7]:
print("duplicate values")
print(anime.duplicated().sum())

duplicate values
0



## 3. Data Cleaning and Preparation


In [8]:
anime["genre"]=anime["genre"].fillna("")
anime["type"]=anime["type"].fillna("Unknown")


## 4. Genre Exploration


In [9]:
# Look at some anime titles and their genres

anime[["name", "genre"]].head(15)

,name,genre
0,Kimi no Na wa.,"Drama, Romance, School, Supernatural"
1,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili..."
2,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S..."
3,Steins;Gate,"Sci-Fi, Thriller"
4,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S..."
5,Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...,"Comedy, Drama, School, Shounen, Sports"
6,Hunter x Hunter (2011),"Action, Adventure, Shounen, Super Power"
7,Ginga Eiyuu Densetsu,"Drama, Military, Sci-Fi, Space"
8,Gintama Movie: Kanketsu-hen - Yorozuya yo Eien...,"Action, Comedy, Historical, Parody, Samurai, S..."
9,Gintama&#039;: Enchousen,"Action, Comedy, Historical, Parody, Samurai, S..."


In [10]:
# Find the most common genres
genre_counts = (anime["genre"].str.split(",").explode().str.strip().value_counts())

In [11]:
genre_counts.head(15)

genre
Comedy           4645
Action           2845
Adventure        2348
Fantasy          2309
Sci-Fi           2070
Drama            2016
Shounen          1712
Kids             1609
Romance          1464
School           1220
Slice of Life    1220
Hentai           1141
Supernatural     1037
Mecha             944
Music             860
Name: count, dtype: int64


## 5. Genre-Based Features


In [12]:
tfidf = TfidfVectorizer(token_pattern=r"[^,]+")

tfidf_matrix = tfidf.fit_transform(anime["genre"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Unique genre features:", len(tfidf.get_feature_names_out()))

TF-IDF matrix shape: (12294, 82)
Unique genre features: 82



## 6. Rating-Based Features


In [13]:
print("Total anime:", len(anime))
print("Missing ratings:", anime["rating"].isna().sum())

print("\nHighest-rated anime:")
print(
    anime[["name", "rating", "members"]].dropna(subset=["rating"])
    .sort_values("rating", ascending=False).head(10)
)

Total anime: 12294
Missing ratings: 230

Highest-rated anime:
                                          name  rating  members
10464  Taka no Tsume 8: Yoshida-kun no X-Files   10.00       13
10400              Spoon-hime no Swing Kitchen    9.60       47
9595                          Mogura no Motoro    9.50       62
0                               Kimi no Na wa.    9.37   200630
9078                              Kahei no Umi    9.33       44
1             Fullmetal Alchemist: Brotherhood    9.26   793665
2                                     Gintama°    9.25   114262
10786          Yakusoku: Africa Mizu to Midori    9.25       53
3                                  Steins;Gate    9.17   673572
4                                Gintama&#039;    9.16   151266


In [14]:
# Average rating across the dataset
C = anime["rating"].mean()

# Minimum number of members required for stronger confidence
m = anime["members"].quantile(0.75)

# Calculate weighted rating
anime["weighted_rating"] = (
    (anime["members"] / (anime["members"] + m)) * anime["rating"]
    + (m / (anime["members"] + m)) * C
)

print("Average rating:", round(C, 2))
print("Minimum popularity threshold:", int(m))

print("\nTop 10 by weighted rating:")
print(
    anime[["name", "rating", "members", "weighted_rating"]].dropna(subset=["rating"])
    .sort_values("weighted_rating", ascending=False).head(10).round(2).to_string(index=False))

Average rating: 6.47
Minimum popularity threshold: 9437

Top 10 by weighted rating:
                                                     name  rating  members  weighted_rating
                                           Kimi no Na wa.    9.37   200630             9.24
                         Fullmetal Alchemist: Brotherhood    9.26   793665             9.23
                                              Steins;Gate    9.17   673572             9.13
                                   Hunter x Hunter (2011)    9.13   425855             9.07
                                                 Gintama°    9.25   114262             9.04
                                     Clannad: After Story    9.06   456749             9.01
                                            Gintama&#039;    9.16   151266             9.00
                                                  Gintama    9.04   336376             8.97
                       Code Geass: Hangyaku no Lelouch R2    8.98   572888             8

In [15]:
# Normalize weighted ratings to a 0–1 scale
min_rating = anime["weighted_rating"].min()
max_rating = anime["weighted_rating"].max()

anime["normalized_rating"] = (
    (anime["weighted_rating"] - min_rating)
    / (max_rating - min_rating)
)


## 7. Story-Based Features (synopsis)


In [16]:
synopsis = pd.read_csv("anime_with_synopsis.csv")

print("Dataset shape:", synopsis.shape)
print("\nColumns:", synopsis.columns.tolist())
print("\nMissing values:")
print(synopsis.isnull().sum())

print("\nFirst 5 anime:")
print(synopsis.head())

Dataset shape: (16214, 5)

Columns: ['MAL_ID', 'Name', 'Score', 'Genres', 'sypnopsis']

Missing values:
MAL_ID       0
Name         0
Score        0
Genres       0
sypnopsis    8
dtype: int64

First 5 anime:
   MAL_ID                             Name Score  \
0       1                     Cowboy Bebop  8.78   
1       5  Cowboy Bebop: Tengoku no Tobira  8.39   
2       6                           Trigun  8.24   
3       7               Witch Hunter Robin  7.27   
4       8                   Bouken Ou Beet  6.98   

                                              Genres  \
0    Action, Adventure, Comedy, Drama, Sci-Fi, Space   
1              Action, Drama, Mystery, Sci-Fi, Space   
2  Action, Sci-Fi, Adventure, Comedy, Drama, Shounen   
3  Action, Mystery, Police, Supernatural, Drama, ...   
4          Adventure, Fantasy, Shounen, Supernatural   

                                           sypnopsis  
0  In the year 2071, humanity has colonized sever...  
1  other day, another bounty—suc

In [17]:
synopsis = synopsis.rename(columns={"sypnopsis": "synopsis"})

# Fill missing synopsis text with an empty string
synopsis["synopsis"] = synopsis["synopsis"].fillna("")

# Remove leading/trailing spaces from anime titles
synopsis["Name"] = synopsis["Name"].str.strip()

print("Missing synopses:", synopsis["synopsis"].isna().sum())
print("Duplicate anime IDs:", synopsis["MAL_ID"].duplicated().sum())
print("Duplicate anime names:", synopsis["Name"].duplicated().sum())

print("\nExample synopsis:")
print(synopsis.loc[0, "synopsis"][:500])

Missing synopses: 0
Duplicate anime IDs: 0
Duplicate anime names: 4

Example synopsis:
In the year 2071, humanity has colonized several of the planets and moons of the solar system leaving the now uninhabitable surface of planet Earth behind. The Inter Solar System Police attempts to keep peace in the galaxy, aided in part by outlaw bounty hunters, referred to as "Cowboys." The ragtag team aboard the spaceship Bebop are two such individuals. Mellow and carefree Spike Spiegel is balanced by his boisterous, pragmatic partner Jet Black as the pair makes a living chasing bounties and 


In [18]:
# Create normalized titles for matching
anime["match_name"] = (
    anime["name"]
    .str.casefold()
    .str.strip()
)

synopsis["match_name"] = (
    synopsis["Name"]
    .str.casefold()
    .str.strip()
)

# Count matching titles
original_titles = set(anime["match_name"])
synopsis_titles = set(synopsis["match_name"])

matched_titles = original_titles & synopsis_titles

print("Anime in original dataset:", len(original_titles))
print("Unique synopsis titles:", len(synopsis_titles))
print("Titles found in both datasets:", len(matched_titles))
print("Original titles without a synopsis match:",
      len(original_titles - synopsis_titles))

Anime in original dataset: 12292
Unique synopsis titles: 16210
Titles found in both datasets: 10044
Original titles without a synopsis match: 2248


In [19]:
# Keep one synopsis row per normalized title
synopsis_unique = synopsis.drop_duplicates(
    subset="match_name"
).copy()

# Keep only titles that exist in the original dataset
matched_synopsis = synopsis_unique[
    synopsis_unique["match_name"].isin(matched_titles)
].copy()

print("Matched anime:", len(matched_synopsis))

Matched anime: 10044


In [20]:
# Map original anime titles to their row positions
original_index = {
    name: i for i, name in enumerate(anime["match_name"])
}

# Map synopsis titles to their row positions in the story matrix
story_index = {
    name: i for i, name in enumerate(matched_synopsis["match_name"])
}

In [21]:
story_tfidf_v2 = TfidfVectorizer(
    stop_words="english",
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_df=0.85
)

story_matrix_v2 = story_tfidf_v2.fit_transform(
    matched_synopsis["synopsis"]
)

print("New matrix shape:", story_matrix_v2.shape)
print("New vocabulary size:",
      len(story_tfidf_v2.get_feature_names_out()))

New matrix shape: (10044, 30000)
New vocabulary size: 30000


In [22]:
story_tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=30000,
    min_df=2,
    sublinear_tf=True,
    max_df=0.90
)

story_matrix_char = story_tfidf_char.fit_transform(
    matched_synopsis["synopsis"]
)

print("Word model shape:", story_matrix_v2.shape)
print("Character model shape:", story_matrix_char.shape)

Word model shape: (10044, 30000)
Character model shape: (10044, 30000)



## 8. Franchise Processing


In [23]:
import html
import re

def clean_title(title):
    title = html.unescape(str(title)).casefold().strip()
    title = re.sub(r"[^a-z0-9\s]", " ", title)
    return re.sub(r"\s+", " ", title).strip()

def get_franchise_key(title):
    title = clean_title(title)

    for franchise in sorted(franchise_names, key=len, reverse=True):
        if title == franchise or title.startswith(franchise + " "):
            return franchise

    return title

In [24]:
franchise_names = [
    "one piece",
    "death note",
    "naruto",
    "bleach",
    "dragon ball",
    "attack on titan",
    "shingeki no kyojin",
    "fullmetal alchemist",
    "sword art online",
    "my hero academia",
    "detective conan",
    "gintama",
    "haikyuu",
    "clannad",
    "code geass",
    "hunter x hunter",
    "steins gate",
    "toriko",
    "digimon",
    "katekyo hitman reborn",
    "tsubasa"
]

In [25]:
for title in [
    "Shingeki no Kyojin",
    "Shingeki no Kyojin OVA",
    "Shingeki no Kyojin Movie 1: Guren no Yumiya",
    "Toriko",
    "Toriko: Jump Super Anime Tour 2009 Special",
]:
    print(title, "->", get_franchise_key(title))

Shingeki no Kyojin -> shingeki no kyojin
Shingeki no Kyojin OVA -> shingeki no kyojin
Shingeki no Kyojin Movie 1: Guren no Yumiya -> shingeki no kyojin
Toriko -> toriko
Toriko: Jump Super Anime Tour 2009 Special -> toriko



## 9. Anime Recommendation Engine


In [26]:
anime_aliases = {
    "attack on titan": "Shingeki no Kyojin",
    "your name": "Kimi no Na wa."
}

In [27]:
# Map each matched anime to its row in the original anime dataset
candidate_names = matched_synopsis["match_name"].tolist()

candidate_original_indices = np.array([
    original_index[name] for name in candidate_names
])

# Add franchise labels to the matched synopsis dataset
matched_synopsis["franchise"] = (
    matched_synopsis["Name"].apply(get_franchise_key)
)

def recommend_anime_modes(title, top_n=10, mode="both"):
    requested_title=title.casefold().strip()
    canonical_title = anime_aliases.get(requested_title,title)
    key = canonical_title.casefold().strip()

    if key not in original_index or key not in story_index:
        print(f"'{title}' must exist in both datasets.")
        return

    original_idx = original_index[key]
    story_idx = story_index[key]
    target_franchise = get_franchise_key(canonical_title)

    # Genre similarity
    genre_scores = cosine_similarity(
        tfidf_matrix[original_idx], tfidf_matrix
    ).flatten()

    # Combined word + character text similarity
    word_scores = cosine_similarity(
        story_matrix_v2[story_idx], story_matrix_v2
    ).flatten()

    char_scores = cosine_similarity(
        story_matrix_char[story_idx], story_matrix_char
    ).flatten()

    story_scores = 0.70 * word_scores + 0.30 * char_scores

    # Align candidates with the original anime dataset
    candidate_genres = genre_scores[candidate_original_indices]

    candidate_ratings = anime.loc[
        candidate_original_indices, "normalized_rating"
    ].fillna(0).to_numpy()

    # Hybrid score: 40% genre + 40% text + 20% rating
    hybrid_scores = (
        0.40 * candidate_genres
        + 0.40 * story_scores
        + 0.20 * candidate_ratings
    )

    candidates = matched_synopsis.copy()
    candidates["genre_similarity"] = candidate_genres
    candidates["story_similarity"] = story_scores
    candidates["hybrid_score"] = hybrid_scores
    candidates["original_idx"] = candidate_original_indices

    # Exclude the exact anime being searched
    candidates = candidates[
        candidates["match_name"] != key
    ].copy()

    if mode == "related":
        candidates = candidates[
            candidates["franchise"] == target_franchise
        ]
    elif mode == "discover":
        candidates = candidates[
            candidates["franchise"] != target_franchise
        ]
    elif mode != "both":
        print("Choose mode='related', 'discover', or 'both'.")
        return

    def top_results(frame):
        frame = frame.sort_values(
            "hybrid_score", ascending=False
        ).head(top_n)

        return frame[
            [
                "Name", "Genres", "Score",
                "genre_similarity", "story_similarity",
                "hybrid_score"
            ]
        ].reset_index(drop=True).round(3)

    if mode == "both":
        print("RELATED ANIME")
        display(top_results(
            candidates[candidates["franchise"] == target_franchise]
        ))

        print("DISCOVER NEW ANIME")
        display(top_results(
            candidates[candidates["franchise"] != target_franchise]
        ))
    else:
        return top_results(candidates)

In [28]:
def recommend_diverse_anime(title, top_n=10):
    results = recommend_anime_modes(
        title, top_n=100, mode="discover"
    )

    if results is None or results.empty:
        return results

    results = results.copy()
    results["franchise"] = results["Name"].apply(get_franchise_key)

    # Sort by hybrid score and keep the best anime per franchise
    results = results.sort_values(
        "hybrid_score", ascending=False
    )

    diverse = results.drop_duplicates(
        subset="franchise", keep="first"
    )

    # If there aren't enough unique franchises, fill remaining slots
    if len(diverse) < top_n:
        remaining = results[
            ~results.index.isin(diverse.index)
        ]
        diverse = pd.concat([
            diverse,
            remaining.head(top_n - len(diverse))
        ])

    return diverse.head(top_n).drop(
        columns="franchise"
    ).reset_index(drop=True)


## 10. Results of the Final Model


In [29]:
recommend_anime_modes("One Piece", mode="both")

RELATED ANIME


,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,One Piece Film: Strong World Episode 0,"Action, Adventure, Comedy, Fantasy, Shounen, S...",8.0,0.918,0.223,0.607
1,One Piece Film: Strong World,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.17,0.873,0.207,0.597
2,One Piece: Episode of Merry - Mou Hitori no Na...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.21,1.000,0.129,0.597
3,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,"Action, Adventure, Comedy, Super Power, Drama,...",7.72,1.000,0.136,0.584
4,One Piece Film: Z,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.18,0.873,0.175,0.583
5,One Piece: Oounabara ni Hirake! Dekkai Dekkai ...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.29,0.918,0.210,0.577
6,One Piece Movie 2: Nejimaki-jima no Daibouken,"Action, Adventure, Comedy, Super Power, Fantas...",7.17,0.918,0.163,0.561
7,One Piece: Episode of Nami - Koukaishi no Nami...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.15,1.000,0.015,0.555
8,One Piece Movie 1,"Action, Adventure, Comedy, Super Power, Fantas...",7.1,0.918,0.152,0.555
9,One Piece Movie 4: Dead End no Bouken,"Action, Adventure, Comedy, Super Power, Fantas...",7.59,0.918,0.110,0.552


DISCOVER NEW ANIME


,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Hunter x Hunter (2011),"Action, Adventure, Fantasy, Shounen, Super Power",9.1,0.777,0.082,0.538
1,Shingeki no Kyojin,"Action, Military, Mystery, Super Power, Drama,...",8.48,0.838,0.048,0.530
2,Shingeki no Kyojin OVA,"Action, Super Power, Drama, Fantasy, Shounen",7.83,0.838,0.092,0.522
3,Hunter x Hunter,"Action, Adventure, Super Power, Fantasy, Shounen",8.42,0.777,0.089,0.517
4,Digimon Frontier,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",7.15,0.873,0.069,0.506
5,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.16,0.786,0.054,0.503
6,Dragon Ball Movie 1: Shen Long no Densetsu,"Action, Adventure, Comedy, Fantasy, Shounen, S...",6.83,0.918,0.033,0.500
7,Hunter x Hunter: Greed Island,"Action, Adventure, Super Power, Fantasy, Shounen",8.21,0.777,0.041,0.487
8,Dragon Ball Movie 4: Saikyou e no Michi,"Action, Adventure, Comedy, Fantasy, Sci-Fi, Sh...",7.15,0.866,0.031,0.486
9,Hunter x Hunter: Greed Island Final,"Action, Adventure, Super Power, Fantasy, Shounen",8.29,0.777,0.031,0.485


In [30]:
recommend_diverse_anime("One Piece", top_n=10)

,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Hunter x Hunter (2011),"Action, Adventure, Fantasy, Shounen, Super Power",9.1,0.777,0.082,0.538
1,Shingeki no Kyojin,"Action, Military, Mystery, Super Power, Drama,...",8.48,0.838,0.048,0.530
2,Digimon Frontier,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",7.15,0.873,0.069,0.506
3,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.16,0.786,0.054,0.503
4,Katekyo Hitman Reborn!,"Action, Comedy, Shounen, Super Power",8.18,0.749,0.042,0.484
5,Toriko,"Action, Adventure, Comedy, Fantasy, Shounen",7.56,0.778,0.064,0.479
6,Naruto: Takigakure no Shitou - Ore ga Eiyuu Da...,"Action, Adventure, Comedy, Shounen, Super Power",6.76,0.856,0.042,0.475
7,Fairy Tail (2014),"Action, Adventure, Comedy, Fantasy, Magic, Sho...",7.73,0.678,0.093,0.472
8,Tsubasa: Tokyo Revelations,"Action, Adventure, Drama, Romance, Fantasy, Sh...",8.31,0.725,0.044,0.471
9,Hai to Gensou no Grimgar,"Action, Adventure, Drama, Fantasy",7.69,0.719,0.066,0.465


In [31]:
recommend_diverse_anime("Death Note", top_n=10)

,Name,Genres,Score,genre_similarity,story_similarity,hybrid_score
0,Higurashi no Naku Koro ni Kai,"Mystery, Psychological, Supernatural, Thriller",8.23,0.899,0.039,0.544
1,Mousou Dairinin,"Mystery, Dementia, Police, Psychological, Supe...",7.68,0.744,0.069,0.471
2,Boku dake ga Inai Machi,"Mystery, Psychological, Supernatural, Seinen",8.37,0.669,0.049,0.465
3,Jigoku Shoujo Mitsuganae,"Mystery, Psychological, Supernatural",7.65,0.741,0.045,0.459
4,Shiki,"Horror, Mystery, Supernatural, Thriller, Vampire",7.79,0.666,0.081,0.454
5,Yojouhan Shinwa Taikei,"Mystery, Comedy, Psychological, Romance",8.61,0.629,0.060,0.450
6,Denpa-teki na Kanojo,"Mystery, Thriller, School",7.58,0.678,0.072,0.446
7,Ergo Proxy,"Psychological, Mystery, Sci-Fi",7.92,0.639,0.082,0.445
8,Monster,"Drama, Horror, Mystery, Police, Psychological,...",8.76,0.595,0.064,0.443
9,Yakushiji Ryouko no Kaiki Jikenbo,"Mystery, Police, Supernatural",7.09,0.743,0.065,0.442


In [32]:
results = recommend_diverse_anime("Death Note", top_n=10).copy()

results["genre_contribution"] = (
    0.40 * results["genre_similarity"]
)

results["text_contribution"] = (
    0.40 * results["story_similarity"]
)

results["rating_contribution"] = (
    results["hybrid_score"]
    - results["genre_contribution"]
    - results["text_contribution"]
)

display(
    results[
        [
            "Name",
            "genre_contribution",
            "text_contribution",
            "rating_contribution",
            "hybrid_score",
        ]
    ].round(3)
)

,Name,genre_contribution,text_contribution,rating_contribution,hybrid_score
0,Higurashi no Naku Koro ni Kai,0.360,0.016,0.169,0.544
1,Mousou Dairinin,0.298,0.028,0.146,0.471
2,Boku dake ga Inai Machi,0.268,0.020,0.178,0.465
3,Jigoku Shoujo Mitsuganae,0.296,0.018,0.145,0.459
4,Shiki,0.266,0.032,0.155,0.454
5,Yojouhan Shinwa Taikei,0.252,0.024,0.174,0.450
6,Denpa-teki na Kanojo,0.271,0.029,0.146,0.446
7,Ergo Proxy,0.256,0.033,0.157,0.445
8,Monster,0.238,0.026,0.179,0.443
9,Yakushiji Ryouko no Kaiki Jikenbo,0.297,0.026,0.119,0.442



## 11. Testing and Validation


In [33]:
test_titles = [
    "Death Note",
    "One Piece",
    "Naruto",
    "Attack on Titan",
    "Your Name"
]

for title in test_titles:
    print(f"\n{'=' * 50}")
    print(f"Recommendations for: {title}")
    print("=" * 50)

    results = recommend_diverse_anime(title, top_n=5)

    if results is not None and not results.empty:
        display(results[["Name", "Score", "hybrid_score"]])
    else:
        print("No recommendations found.")


Recommendations for: Death Note


,Name,Score,hybrid_score
0,Higurashi no Naku Koro ni Kai,8.23,0.544
1,Mousou Dairinin,7.68,0.471
2,Boku dake ga Inai Machi,8.37,0.465
3,Jigoku Shoujo Mitsuganae,7.65,0.459
4,Shiki,7.79,0.454



Recommendations for: One Piece


,Name,Score,hybrid_score
0,Hunter x Hunter (2011),9.1,0.538
1,Shingeki no Kyojin,8.48,0.530
2,Digimon Frontier,7.15,0.506
3,Dragon Ball Z,8.16,0.503
4,Katekyo Hitman Reborn!,8.18,0.484



Recommendations for: Naruto


,Name,Score,hybrid_score
0,Boruto: Naruto the Movie,7.5,0.652
1,Boruto: Naruto the Movie - Naruto ga Hokage ni...,7.4,0.577
2,The Last: Naruto the Movie,7.76,0.572
3,Dragon Ball Z,8.16,0.530
4,Katekyo Hitman Reborn!,8.18,0.500



Recommendations for: Attack on Titan


,Name,Score,hybrid_score
0,One Piece,8.52,0.530
1,Saint Seiya: Meiou Hades Elysion-hen,7.59,0.506
2,Final Fantasy VII: Advent Children,7.76,0.492
3,Hunter x Hunter (2011),9.1,0.489
4,Final Fantasy VII: Advent Children Complete,7.96,0.483



Recommendations for: Your Name


,Name,Score,hybrid_score
0,Wind: A Breath of Heart (TV),6.0,0.531
1,Wind: A Breath of Heart OVA,6.24,0.527
2,Kokoro ga Sakebitagatterunda.,7.96,0.524
3,Angel Beats!: Another Epilogue,7.49,0.514
4,Harmonie,7.31,0.512


In [34]:
results = recommend_anime_modes(
    "Attack on Titan",
    top_n=10,
    mode="discover"
)

display(results[["Name", "hybrid_score"]])

,Name,hybrid_score
0,One Piece,0.530
1,Saint Seiya: Meiou Hades Elysion-hen,0.506
2,Final Fantasy VII: Advent Children,0.492
3,One Piece: Episode of Merry - Mou Hitori no Na...,0.491
4,One Piece: Episode of Nami - Koukaishi no Nami...,0.490
5,Hunter x Hunter (2011),0.489
6,Final Fantasy VII: Advent Children Complete,0.483
7,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,0.479
8,Guilty Crown,0.472
9,One Piece: Romance Dawn Story,0.460


In [35]:
for title in [
    "Death Note",
    "One Piece",
    "Naruto",
    "Attack on Titan",
    "Your Name"
]:
    results = recommend_diverse_anime(title, top_n=5)

    if results is not None and not results.empty:
        franchises = results["Name"].apply(get_franchise_key)

        print(
            f"{title}: "
            f"{len(results)} recommendations, "
            f"{franchises.nunique()} unique franchises"
        )

Death Note: 5 recommendations, 5 unique franchises
One Piece: 5 recommendations, 5 unique franchises
Naruto: 5 recommendations, 5 unique franchises
Attack on Titan: 5 recommendations, 5 unique franchises
Your Name: 5 recommendations, 5 unique franchises
